In [ ]:
import json
import os
import time

import bibtexparser
import flatdict as fd
import numpy as np
import pandas as pd
import requests

from pynxtools_em.examples.oasisb_bibliography import get_bibliographical_metadata
from pynxtools_em.examples.oasisb_openalex import get_data_for_doi_from_openalex
from pynxtools_em.examples.oasisb_utils import get_project_id

rng = np.random.default_rng(seed=42)

print(os.getcwd())
with open("source_directory.txt") as fp:
    src_directory = f"{fp.readline().strip().replace('/', os.sep)}"
print(src_directory)

In [ ]:
spread_sheet_of_all_projects = pd.read_excel(
    f"{src_directory}{os.sep}aaa_legacy_data.ods",
    sheet_name="aaa_legacy_data",
    engine="odf",
    dtype="str",
).fillna("")
with open(f"{src_directory}{os.sep}aaa_legacy_data.bib") as fp:
    bib = bibtexparser.load(fp).entries_dict
project_range: tuple[int, int] = (1, 880)

## Query for each project bibliographical metadata from OpenAlex

Querying metadata in addition to the DOI allows to cross-check authors, institutions, and copyright relevant details.

In [ ]:
api_queries_cnt = 0
api_queries_max = 10
for row in spread_sheet_of_all_projects.itertuples(index=True):
    if row.project_name != "" and row.legal in ("0", "1") and row.use in ("0", "1"):
        # row.parse == 2 if really only cross-ref data if available if dataset is CC0-1.0 or CC-BY-4.0
        # row.parse in (1, 2) if also allowing locally shared datasets, these will not be uploaded to any public deployment though
        if project_range[0] <= int(row.project_name) <= project_range[1]:
            project_id = get_project_id(f"{row.project_name}")
            data_and_paper = get_bibliographical_metadata(bib, project_id)

            n_queries = get_data_for_doi_from_openalex(bib, data_and_paper)

            api_queries_cnt += n_queries  # sleep only when necessary
            if api_queries_cnt >= api_queries_max:
                sleep = float(rng.uniform(1, 10))
                print(f"Sleeping for {sleep}s")
                time.sleep(sleep)
                api_queries_cnt = 0
print("Batch querying queue done")

***

## Normalize author field cross-checking against OpenAlex

Often author names are abbreviated, e.g. Breen, A. J. instead of Breen, Andrew John.
Here we remove abbreviations.

In [ ]:
count: int = 0
for row in spread_sheet_of_all_projects.itertuples(index=True):
    if row.project_name != "" and row.legal in ("0", "1") and row.use in ("0", "1"):
        # row.parse == 2 if really only cross-ref data if available if dataset is CC0-1.0 or CC-BY-4.0
        # row.parse in (1, 2) if also allowing locally shared datasets, these will not be uploaded to any public deployment though
        if row.legal == "1" and row.use == "1":
            project_id = get_project_id(row.project_name)
            data_and_paper: list[str] = get_bibliographical_metadata(
                bib,
                project_id,
                verbose=False,
            )
            # print(f"{project_id}, {bib[data_and_paper[0]]['author']}")
            openalex_authors: list[str] = []
            openalex_titles: list[str] = []
            openalex_affil: int = 0
            openalex_file = (
                f"{os.getcwd()}{os.sep}openalex{os.sep}{data_and_paper[0]}.json"
            )
            try:
                if os.path.isfile(openalex_file):
                    with open(openalex_file, encoding="utf-8") as fp:
                        openalex = fd.FlatDict(json.load(fp), "/")
                        if "title" in openalex:
                            openalex_titles.append(openalex["title"].strip())
                        if "authorships" in openalex:
                            for dct in openalex["authorships"]:
                                flat = fd.FlatDict(dct, "/")
                                if "institutions" in flat:
                                    if len(flat["institutions"]) > 0:
                                        openalex_affil = 1
                                if "raw_author_name" in flat:
                                    openalex_authors.append(
                                        flat["raw_author_name"].strip()
                                    )
            except TypeError:
                pass

            count += 1
            if openalex_affil == 0:
                print(f"{project_id}, {data_and_paper[0]}")
            continue

            # if int(project_id) in take:
            #     new_bib[data_and_paper[0]]["author"] = " and ".join(openalex_authors)
            # continue
            # 308 <= x <= 328
            # if int(project_id) < 870:
            #     continue
            # if int(project_id) > 880:
            #     break
            bibtex_authors: list[str] = []
            bibtex_titles: list[str] = []
            if data_and_paper[0] != "" and "author" in bib[data_and_paper[0]]:
                if "title" in bib[data_and_paper[0]]:
                    bibtex_titles.append(bib[data_and_paper[0]]["title"].strip())
                for bibtex_author in [
                    value.strip()
                    for value in bib[data_and_paper[0]]["author"].split("and")
                ]:
                    bibtex_authors.append(bibtex_author)

            # if len(openalex_titles) == 0 and len(bibtex_titles) == 0:
            #     #  if openalex_titles[0] != bibtex_titles[0]:
            print(
                f"{project_id}, {bib[data_and_paper[0]]['author']}"
            )  # {openalex_titles},

            """
            if openalex_authors != bibtex_authors:
                print(project_id)
                n = min(len(openalex_authors), len(bibtex_authors))
                for idx in range(0, n):
                    if openalex_authors[idx] != bibtex_authors[idx]:
                        print(f"\t{openalex_authors[idx]} != {bibtex_authors[idx]}")
                print(f"\t{openalex_authors[n:]}")
                print(f"\t{bibtex_authors[n:]}")
                print(f"{project_id}, {openalex_authors}")
                print(f"{project_id}, {bibtex_authors}")
            """
            del openalex_authors, bibtex_authors
print(f"Batch queue done {count}")

***

In [ ]:
from bibtexparser.bwriter import BibTexWriter

In [ ]:
db = BibDatabase()
db.entries = [new_bib]
with open(
    f"{src_directory}{os.sep}aaa_legacy_bib_mod.bib", "w", encoding="utf-8"
) as bibfile:
    bibfile.write(writer.write(db))

In [ ]:
print(type(new_bib))
import pycountry
import requests

openalex_id = "I204778367"
data = requests.get(f"https://api.openalex.org/institutions/{openalex_id}").json()
city = data["geo"]["city"]
iso2 = data["geo"]["country_code"]
# convert ISO2 -> ISO3
iso3 = pycountry.countries.get(alpha_2=iso2).alpha_3
print(city, iso3)